In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/engage-2-value-from-clicks-to-conversions/sample_submission.csv
/kaggle/input/engage-2-value-from-clicks-to-conversions/train_data.csv
/kaggle/input/engage-2-value-from-clicks-to-conversions/test_data.csv


# Engage2Value: From Clicks to Conversions

### The goal of this competition is to predict a customer’s purchase value based on their multi-session behavior across digital touchpoints.
### The dataset captures anonymized user interactions such as browser types, traffic sources, device details, and geographical indicators. 

In [2]:
from sklearn.ensemble import GradientBoostingRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# 1. Loading The Dataset

In [3]:
df = pd.read_csv("/kaggle/input/engage-2-value-from-clicks-to-conversions/train_data.csv") 

# 2. Identifying data types of different columns

In [ ]:
df.info()

#  General Info

- **Total rows (sessions):** 116,023  
- **Total columns (features):** 52  

### Data type spread:
- `object` (categorical/text): 37 columns  
- `float64` (decimal numbers): 5 columns  
- `int64` (whole numbers): 9 columns  
- `bool`: 1 column  

---

#  Key Column Descriptions

##  Target Variable

- **`purchaseValue`** (`float64`):  
  Total value of purchase made during the session. This is likely your **target variable** for a regression task.

---

##  Traffic & Source Info

- **`trafficSource.isTrueDirect`, `trafficSource.keyword`, `trafficSource.campaign`, etc.**:  
  Provide info about how the user arrived at the site — whether directly, via ads, or organic search.

- **`trafficSource.adwordsClickInfo.*`**:  
  Contains ad-related metadata like ad slot, network type, if it was a video ad, etc.  
  These have **many missing values** — e.g., only 4,281 non-null rows.

---

##  Device Information

- **`deviceType`, `device.isMobile`, `device.operatingSystemVersion`, etc.**:  
  Metadata about the user's device, OS, browser, screen size, flash version, etc.

---

##  Geographic Info

- **`geoNetwork.continent`, `geoNetwork.region`, `geoNetwork.city`, `geoCluster`, etc.**:  
  Indicates user location based on IP or location services.

---

##  User/Session Info

- **`userId`, `sessionId`, `sessionNumber`, etc.**:  
  Identifies users and their sessions.

- **`new_visits`**:  
  Indicates whether the session was from a new user (has missing values — 80,518 non-null).

- **`gclIdPresent`**:  
  A binary flag — possibly indicating if the session came from a Google Click ID (Ad tracking).

---

##  Session Behavior

- **`pageViews`, `totals.visits`, `totals.bounces`, `totalHits`**:  
  Metrics on what the user did during the session — how many pages they viewed, whether they bounced (left after 1 page), etc.  
  Some of these have **missing values** (e.g., `bounces` has only 47,148 non-null).

---

##  Temporal Info

- **`date`**:  
  Likely in `YYYYMMDD` integer format.

- **`sessionStart`**:  
  Unix timestamp or other session start indicator.


# 3.Identifying and handling the missing values.

In [4]:
df = df.replace('(not set)', np.nan)
df = df.replace('not available in demo dataset', np.nan)
df = df.replace('(none)', np.nan)
df = df.replace('(not provided)', np.nan)

/tmp/ipykernel_37/2025767729.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('not available in demo dataset', np.nan)


In [ ]:
df.isnull().sum()

In [8]:
?pd

Type:        module
String form: <module 'pandas' from '/usr/local/lib/python3.11/dist-packages/pandas/__init__.py'>
File:        /usr/local/lib/python3.11/dist-packages/pandas/__init__.py
Docstring:  
pandas - a powerful data analysis and manipulation library for Python

**pandas** is a Python package providing fast, flexible, and expressive data
structures designed to make working with "relational" or "labeled" data both
easy and intuitive. It aims to be the fundamental high-level building block for
doing practical, **real world** data analysis in Python. Additionally, it has
the broader goal of becoming **the most powerful and flexible open source data
analysis / manipulation tool available in any language**. It is already well on
its way toward this goal.

Main Features
-------------
Here are just a few of the things that pandas does well:

  - Easy handling of missing data in floating point as well as non-floating
    point data.
  - Size mutability: columns can be inserted and de

In [17]:
print(df.nunique())

for col in df.columns[df.nunique()==1]:
    print(f"\nColumn: {col}")
    print(df[col].unique())

for col in df.columns[df.nunique()==2]:
    print(f"\nColumn: {col}")
    print(df[col].unique())
df.loc(df[1][1])


trafficSource.isTrueDirect                           1
purchaseValue                                     6766
browser                                             33
device.screenResolution                              0
trafficSource.adContent                             53
trafficSource.keyword                              565
screenSize                                           1
geoCluster                                           5
trafficSource.adwordsClickInfo.slot                  3
device.mobileDeviceBranding                          0
device.mobileInputSelector                           0
userId                                          100499
trafficSource.campaign                              27
device.mobileDeviceMarketingName                     0
geoNetwork.networkDomain                             3
gclIdPresent                                         2
device.operatingSystemVersion                        0
sessionNumber                                      230
device.fla

KeyError: 1

In [6]:
missing = df.isnull().mean().sort_values(ascending=False)
print(type(missing))
missing[missing > 0]

<class 'pandas.core.series.Series'>


device.mobileDeviceMarketingName                1.000000
device.mobileDeviceBranding                     1.000000
device.browserSize                              1.000000
geoNetwork.networkLocation                      1.000000
device.flashVersion                             1.000000
device.operatingSystemVersion                   1.000000
device.mobileDeviceModel                        1.000000
device.mobileInputSelector                      1.000000
browserMajor                                    1.000000
device.screenColors                             1.000000
device.browserVersion                           1.000000
device.language                                 1.000000
device.screenResolution                         1.000000
trafficSource.adContent                         0.974462
trafficSource.adwordsClickInfo.slot             0.963102
trafficSource.adwordsClickInfo.page             0.963102
trafficSource.adwordsClickInfo.isVideoAd        0.963102
trafficSource.adwordsClickInfo.

In [20]:
df['totals.bounces']


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


0         1.0
1         1.0
2         NaN
3         1.0
4         NaN
         ... 
116018    NaN
116019    1.0
116020    1.0
116021    NaN
116022    NaN
Name: totals.bounces, Length: 116023, dtype: float64

In [21]:
df['totals.bounces'].astype(str).str.strip()

0         1.0
1         1.0
2         nan
3         1.0
4         nan
         ... 
116018    nan
116019    1.0
116020    1.0
116021    nan
116022    nan
Name: totals.bounces, Length: 116023, dtype: object

In [ ]:
df['trafficSource.isTrueDirect'] = df['trafficSource.isTrueDirect'].fillna(False)
df['geoNetwork.city'] = df['geoNetwork.city'].fillna('unknown')
df['new_visits'] = df['new_visits'].fillna("0")
df['totals.bounces'] = df['totals.bounces'].fillna("0")
df['trafficSource.isTrueDirect'] = df['trafficSource.isTrueDirect'].astype(bool)

df['totals.bounces'] = df['totals.bounces'].astype(str).str.strip()
df['totals.bounces'] = df['totals.bounces'].map({'0': False, '1.0': True})

df['new_visits'] = df['new_visits'].astype(str).str.strip()
df['new_visits'] = df['new_visits'].map({'0': False, '1.0': True})                                        
                                                                          
df['gclIdPresent'] = df['gclIdPresent'].astype(str).str.strip()
df['gclIdPresent'] = df['gclIdPresent'].map({'0': False, '1': True})                                        
                                                                   

In [ ]:
df.dropna(axis=1, thresh=int(0.10 * len(df)), inplace=True)

In [ ]:
constant_cols = [col for col in df.columns if df[col].nunique(dropna=False) == 1]
print("Columns with single unique value:", constant_cols)
df.drop(columns=constant_cols, inplace=True)

In [ ]:
high_card_cols = ['sessionId']
df.drop(columns=high_card_cols, inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:

df.info()

# 4. Correlation Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr_matrix = df.corr(numeric_only=True)


plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix Heatmap")
plt.show()


# 5.Identifying and handling duplicate values

In [14]:
df.loc[df.duplicated()]

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,trafficSource.isTrueDirect,purchaseValue,browser,device.screenResolution,trafficSource.adContent,trafficSource.keyword,screenSize,geoCluster,trafficSource.adwordsClickInfo.slot,device.mobileDeviceBranding,...,device.language,deviceType,userChannel,device.browserVersion,totalHits,device.screenColors,sessionStart,geoNetwork.continent,device.isMobile,new_visits
2443,True,0.0,Opera,NaN,NaN,NaN,medium,Region_2,NaN,NaN,...,NaN,desktop,Direct,NaN,1,NaN,1489424292,Americas,False,1.0
9906,NaN,56880000.0,Chrome,NaN,NaN,NaN,medium,Region_3,NaN,NaN,...,NaN,desktop,Organic Search,NaN,46,NaN,1495722033,Americas,False,1.0
14779,NaN,54560000.0,Chrome,NaN,NaN,NaN,medium,Region_4,NaN,NaN,...,NaN,desktop,Organic Search,NaN,30,NaN,1493749502,Americas,False,1.0
15765,True,53950000.0,Chrome,NaN,NaN,NaN,medium,Region_5,NaN,NaN,...,NaN,desktop,Direct,NaN,30,NaN,1495653538,Americas,False,NaN
19071,True,0.0,Chrome,NaN,NaN,NaN,medium,Region_5,NaN,NaN,...,NaN,desktop,Direct,NaN,1,NaN,1470070230,Americas,False,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114188,NaN,33590000.0,Chrome,NaN,NaN,NaN,medium,Region_1,NaN,NaN,...,NaN,desktop,Social,NaN,17,NaN,1493661267,Americas,False,NaN
115255,NaN,8990000.0,Safari,NaN,NaN,NaN,medium,Region_3,NaN,NaN,...,NaN,desktop,Organic Search,NaN,31,NaN,1501545464,Americas,False,1.0
115304,True,65960000.0,Chrome,NaN,NaN,NaN,medium,Region_3,NaN,NaN,...,NaN,desktop,Direct,NaN,10,NaN,1480356470,Americas,False,1.0
115766,True,18150000.0,Chrome,NaN,NaN,NaN,medium,Region_3,NaN,NaN,...,NaN,desktop,Organic Search,NaN,27,NaN,1473260512,Americas,False,NaN


In [ ]:
df = df.drop_duplicates()

# 6. Feature Engineering

In [ ]:
cat_cols = ['browser', 'os', 'geoNetwork.subContinent', 'deviceType', 'userChannel']

for col in cat_cols:
    plt.figure(figsize=(10, 5))
    order = df.groupby(col)['purchaseValue'].mean().sort_values(ascending=False).index
    sns.barplot(x=col, y='purchaseValue', data=df, order=order)
    plt.xticks(rotation=45, ha='right')
    plt.title(f'Mean PurchaseValue by {col}')
    plt.tight_layout()
    plt.show()


In [ ]:
top_n = 10
col = 'geoNetwork.city'
top_cities = df[col].value_counts().nlargest(top_n).index

plt.figure(figsize=(10, 5))
sns.boxplot(x=col, y='purchaseValue', data=df[df[col].isin(top_cities)])
plt.title(f'{col} vs PurchaseValue (Top {top_n})')
plt.xticks(rotation=45)
plt.show()

In [ ]:
sns.lmplot(x='pageViews', y='purchaseValue', data=df, scatter_kws={'alpha':0.3})
plt.title("PageViews vs PurchaseValue")

In [ ]:
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
df['sessionStart'] = pd.to_datetime(df['sessionStart'], unit='s')

df['date'] = df['sessionStart'].dt.date

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek
df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6]).astype(int)
df['hour'] = df['sessionStart'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df = df.drop(['date', 'sessionStart','hour'], axis=1)

In [ ]:
df['pag_per_hit']=df['pageViews']/df['totalHits']

df['session_efficiency'] = df['totalHits'] / (df['sessionNumber'] + 1)

mean_purchase_by_city = df.groupby('geoNetwork.city')['purchaseValue'].mean()
df['city_mean_purchase'] = df['geoNetwork.city'].map(mean_purchase_by_city)

df['isReturningUser'] = df['sessionNumber'] >1

In [ ]:
df.nunique()

In [ ]:
corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix Heatmap")
plt.show()


In [ ]:
X = df.drop("purchaseValue", axis=1) 
y = df['purchaseValue']

# 7. Identifying and handling outliers.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

features_to_check = ['pageViews', 'totalHits', 'sessionNumber','pag_per_hit','session_efficiency']

print(df[features_to_check].describe(percentiles=[.25, .5, .75, .90, .95, .99]))

plt.figure(figsize=(15, 5))
for i, feature in enumerate(features_to_check):
    plt.subplot(1, 5, i+1)
    sns.boxplot(x=df[feature])
    plt.title(f'Boxplot: {feature}')
plt.tight_layout()
plt.show()


In [ ]:
for col in ['pageViews', 'totalHits', 'sessionNumber','pag_per_hit','session_efficiency']:
    lower = df[col].quantile(0.04)
    upper = df[col].quantile(0.96)
    df = df[(df[col] >= lower) & (df[col] <= upper)]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Select the numerical features
features_to_check = ['pageViews', 'totalHits', 'sessionNumber','pag_per_hit','session_efficiency']

# Basic stats
print(df[features_to_check].describe(percentiles=[.25, .5, .75, .90, .95, .99]))

# Boxplot to visualize outliers
plt.figure(figsize=(15, 5))
for i, feature in enumerate(features_to_check):
    plt.subplot(1, 5, i+1)
    sns.boxplot(x=df[feature])
    plt.title(f'Boxplot: {feature}')
plt.tight_layout()
plt.show()

# 8. .Scaling of Numerical features and Encoding of Categorical features.

In [ ]:
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
bool_cols = X.select_dtypes(include=['bool']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

onehot_cols = [col for col in cat_cols if df[col].nunique() <= 35]

target_encode_cols = [col for col in cat_cols if df[col].nunique() > 35]

df[bool_cols] = df[bool_cols].astype(int)


print({
    "onehot_cols": onehot_cols,
    "target_encode_cols": target_encode_cols,
    "num_cols": num_cols,
    "bool_cols": bool_cols
})


In [ ]:
for col in target_encode_cols:
    X[col] = X[col].astype(str)
for col in onehot_cols:
    X[col] = X[col].fillna("missing").astype(str)
X[num_cols] = X[num_cols].apply(pd.to_numeric, errors='coerce')
for col in bool_cols:
    X[col] = X[col].astype(bool)


In [ ]:
for col in bool_cols:
    X[col] = X[col].astype(float)

print(X[bool_cols].dtypes)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,       # 20% test data
    random_state=42,     # for reproducibility
)

In [ ]:
X.shape

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from category_encoders import CountEncoder  # or use FrequencyEncoder
from category_encoders import CatBoostEncoder
from category_encoders import TargetEncoder
from sklearn.preprocessing import OrdinalEncoder
# Pipelines
onehot_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

target_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  TargetEncoder())# normalize=True = frequency
])

num_pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=3)),
    ('scaler', MinMaxScaler())
])

bool_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# ColumnTransformer
full_pipeline = ColumnTransformer([
    ('onehot', onehot_pipeline, onehot_cols),
    ('target', target_pipeline, target_encode_cols),
    ('num', num_pipeline, num_cols),
    ('bool', bool_pipeline, bool_cols),
])

# Final preprocessing pipeline
preprocess_pipeline = Pipeline([
    ('transform', full_pipeline)
])


In [ ]:
X_train.describe()

In [ ]:
preprocess_pipeline.set_output(transform="pandas")
X_processed = preprocess_pipeline.fit_transform(X_train,y_train)

In [ ]:
df_clean = X_processed.copy()
print(df_clean.columns)
df_clean.shape
#df_clean = pd.DataFrame(X_processed, columns=final_columns)

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(df_clean.shape, y.shape)
mi_scores = mutual_info_regression(df_clean, y_train)

mi_series = pd.Series(mi_scores, index=df_clean.columns).sort_values(ascending=False)
print("🔍 Top  features by mutual information:")
print(mi_series.head(60))

top_n = 60
top_features = mi_series.head(top_n)

plt.figure(figsize=(15, 16))
sns.barplot(x=top_features.values, y=top_features.index, palette='viridis')
plt.title(f"Top {top_n} Features by Mutual Information")
plt.xlabel("Mutual Information Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
top_n = 41
selected_features = mi_series.head(top_n).index.tolist()
X_selected = df_clean[selected_features]

In [ ]:
X_selected.nunique()

# 9.Model Building

In [ ]:
X_processed_test = preprocess_pipeline.transform(X_test)

df_clean_test = pd.DataFrame(X_processed_test, columns=df_clean.columns)
df_clean_test = df_clean_test[selected_features]

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score


models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGB regressor": XGBRegressor()
}

for name, model in models.items():
    model.fit(X_selected, y_train)
    
    y_train_pred = model.predict(X_selected)
    y_train_pred[y_train_pred < 0] = 0
    
    y_test_pred = model.predict(df_clean_test)
    y_test_pred[y_test_pred < 0] = 0
    
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    print(f"{name}:")
    print(f"  Train R² Score: {train_r2:.4f}")
    print(f"  Test R² Score:  {test_r2:.4f}")
    print("-" * 40)


# 10. Hyper Parameters Tuning for XgbRegressor and RandomForest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, mean_absolute_error

# xgbregressor tuning
"""
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
param_grid = {
    'n_estimators': [400, 500, 600, 700, 800],
    'learning_rate': [0.01, 0.03, 0.05],
    'max_depth': [5, 6, 7, 8],
    'min_child_weight': [3, 4, 5, 6],
    'gamma': [1, 2, 3, 4, 5],
    'subsample': [0.6, 0.7, 0.8, 0.85],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.85],
    'reg_alpha': [5, 10, 15, 20],
    'reg_lambda': [2, 5, 10, 15, 20]
}

model = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)

rs = RandomizedSearchCV(
    model, param_distributions=param_grid,
    n_iter=50,  # Increase for a more thorough search
    scoring='r2',
    cv=5,  # 5-fold cross-validation
    verbose=2,
    random_state=42,
    n_jobs=-1
)
rs.fit(X_selected, y_train)
print("Best params:", rs.best_params_)
print("Best R² CV score:", rs.best_score_)"""

In [ ]:
"""
param_grid = {
    'n_estimators': [100, 200, 300, 400, 500, 600, 700, 800],
    'max_depth': [None, 10, 15, 20, 25, 30],
    'min_samples_split': [2, 3, 4, 5],
    'min_samples_leaf': [1, 2, 3, 4],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

# Instantiate the model
model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# Run RandomizedSearchCV
rs = RandomizedSearchCV(
    model,
    param_distributions=param_grid,
    n_iter=50,  # You can increase for more thorough tuning
    scoring='r2',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Fit on training data
rs.fit(X_selected, y_train)

# Print best params and score
print("Best params:", rs.best_params_)
print("Best R² CV score:", rs.best_score_)

"""

In [ ]:
"""
best_params = {
    'n_estimators': 700,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'max_depth': 25,
    'bootstrap': False
}

# Initialize final model with best params
best_model = RandomForestRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_selected, y_train)
    
y_train_pred = best_model.predict(X_selected)
y_train_pred[y_train_pred < 0] = 0
    
y_test_pred = best_model.predict(df_clean_test)
y_test_pred[y_test_pred < 0] = 0
    
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
    

print(f"  Train R² Score: {train_r2:.4f}")
print(f"  Test R² Score:  {test_r2:.4f}")

"""

# 11. Final Submission

In [ ]:

preprocess_pipeline.set_output(transform="pandas")

X_processed = preprocess_pipeline.fit_transform(X,y)

X_selected=X_processed[selected_features]

best_params = {
    'n_estimators': 700,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'max_depth': 25,
    'bootstrap': False
}

model = RandomForestRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1
)

model.fit(X_selected, y.to_numpy().ravel())

y_train_pred = model.predict(X_selected)

y_train_pred[y_train_pred < 0] = 0
y_train_pred= y_train_pred.ravel()    
mse = mean_squared_error(y, y_train_pred)
r2 = r2_score(y, y_train_pred)
print("R² train score:", r2)
print("RMSE train (scaled):", mse ** 0.5)


In [ ]:
 X_test = pd.read_csv("/kaggle/input/engage-2-value-from-clicks-to-conversions/test_data.csv") 

In [ ]:
X_test = X_test.replace('(not set)', np.nan)
X_test = X_test.replace('not available in demo dataset', np.nan)
X_test = X_test.replace('(none)', np.nan)
X_test = X_test.replace('(not provided)', np.nan)

X_test['trafficSource.isTrueDirect'] = X_test['trafficSource.isTrueDirect'].fillna(False)
X_test['geoNetwork.city'] = X_test['geoNetwork.city'].fillna('unknown')
X_test['new_visits'] = X_test['new_visits'].fillna("0")
X_test['totals.bounces'] = X_test['totals.bounces'].fillna("0")
X_test['trafficSource.isTrueDirect'] = X_test['trafficSource.isTrueDirect'].astype(bool)
X_test['totals.bounces'] = X_test['totals.bounces'].astype(str).str.strip()
X_test['totals.bounces'] = X_test['totals.bounces'].map({'0': False, '1.0': True})

X_test['pag_per_hit']=X_test['pageViews']/X_test['totalHits']

X_test['session_efficiency'] = X_test['totalHits'] / (X_test['sessionNumber'] + 1)

X_test['isReturningUser'] = X_test['sessionNumber'] >1


X_test['city_mean_purchase'] = X_test['geoNetwork.city'].map(mean_purchase_by_city)


X_test['city_mean_purchase'] = X_test['city_mean_purchase'].fillna(mean_purchase_by_city.mean())



X_test['new_visits'] = X_test['new_visits'].astype(str).str.strip()
X_test['new_visits'] = X_test['new_visits'].map({'0': False, '1.0': True})                                        
                                                                          
X_test['gclIdPresent'] = X_test['gclIdPresent'].astype(str).str.strip()
X_test['gclIdPresent'] = X_test['gclIdPresent'].map({'0': False, '1': True})                                        
                                                                   

In [ ]:
for col in bool_cols:
    X_test[col] = X_test[col].fillna(0)
    X_test[col] = X_test[col].astype(float)

X_test[bool_cols] = X_test[bool_cols].round().astype(int)

In [ ]:
X_test['date'] = pd.to_datetime(X_test['date'], errors='coerce')
X_test['sessionStart'] = pd.to_datetime(X_test['sessionStart'], unit='s')
X_test['year'] = X_test['date'].dt.year
X_test['month'] = X_test['date'].dt.month
X_test['day'] = X_test['date'].dt.day
X_test['dayofweek'] = X_test['date'].dt.dayofweek
X_test['is_weekend'] = X_test['date'].dt.dayofweek.isin([5, 6]).astype(int)
X_test['hour'] = X_test['sessionStart'].dt.hour
X_test['hour_sin'] = np.sin(2 * np.pi * X_test['hour'] / 24)
X_test['hour_cos'] = np.cos(2 * np.pi * X_test['hour'] / 24)
X_test = X_test.drop(['date', 'sessionStart','hour'], axis=1)

In [ ]:
X_processed_test = preprocess_pipeline.transform(X_test)

df_clean_test = pd.DataFrame(X_processed_test, columns=df_clean.columns)
df_clean_test = df_clean_test[selected_features]

y_pred_scaled = model.predict(df_clean_test).reshape(-1, 1).ravel()
y_pred_scaled[y_pred_scaled < 0] = 0

X_processed_test

In [ ]:
submission = pd.DataFrame({"id": range(0,X_test.shape[0]), "purchaseValue": y_pred_scaled}) 
submission.to_csv('submission.csv',index=False)